In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Train.csv")
print(df.shape)           # (68654, 16)
print(df.target.mean())   # 0.0183 → 1.83% defaulted


(68654, 16)
0.0183237684621435


In [7]:
num = ["Total_Amount", "duration", "Lender_portion_Funded"]
cat = ["New_versus_Repeat", "loan_type", "lender_id", "country_id"]

X = df[num + cat].copy()
X["lender_id"] = X["lender_id"].astype(str)   # it's an ID, not a quantity
y = df["target"]
groups = df["customer_id"]                    # used to split, not as a feature


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

pre = ColumnTransformer([
    ("num", Pipeline([
        ("log", FunctionTransformer(np.log1p)),  # tame huge amounts
        ("scale", StandardScaler()),             # mean 0, std 1
    ]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),  # categories → 0/1 columns
])


In [9]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000)),
])


In [10]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

results = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    model.fit(X.iloc[tr], y.iloc[tr])                # learn on 4/5
    p = model.predict_proba(X.iloc[va])[:, 1]        # predict the hidden 1/5
    results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y.iloc[va], p),
        "pr_auc": average_precision_score(y.iloc[va], p),
    })

results = pd.DataFrame(results)
print(results.round(3))
print(results.mean().round(3))


   fold  roc_auc  pr_auc
0     0    0.794   0.167
1     1    0.812   0.205
2     2    0.785   0.181
3     3    0.796   0.228
4     4    0.774   0.191
fold       2.000
roc_auc    0.792
pr_auc     0.195
dtype: float64


In [11]:
df2 = pd.read_parquet("../data/processed/features.parquet")
df2 = df2[df2.split == "train"].reset_index(drop=True)   # Test rows have no answers
print(df2.shape)   # (68654, 31): same loans, more columns


(68654, 31)


In [12]:
num2 = ["log_amount_usd", "log_amount_local", "duration", "Lender_portion_Funded",
        "is_new_borrower", "n_lenders_on_loan",
        "n_past_loans", "days_since_last_loan", "n_past_labelled", "n_past_defaults",
        "past_default_rate", "days_since_last_default",
        "disb_month", "disb_dow", "disb_dom",
        "macro_inflation", "macro_fx", "macro_deposit_rate", "macro_unemployment"]
cat2 = ["lender_id", "loan_type"]
X2 = df2[num2 + cat2]
y2 = df2["target"].astype(int)


In [13]:
from sklearn.impute import SimpleImputer

pre2 = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ]), num2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat2),
])
model2 = Pipeline([("pre", pre2),
                   ("lr", LogisticRegression(class_weight="balanced", max_iter=2000))])


In [14]:
results2 = []
for fold in range(5):
    tr = df2.fold_group != fold
    va = df2.fold_group == fold
    model2.fit(X2[tr], y2[tr])
    p = model2.predict_proba(X2[va])[:, 1]
    results2.append({"fold": fold,
                     "roc_auc": roc_auc_score(y2[va], p),
                     "pr_auc": average_precision_score(y2[va], p)})
results2 = pd.DataFrame(results2)
print(results2.round(3))
print(results2.mean().round(3))


   fold  roc_auc  pr_auc
0     0    0.833   0.163
1     1    0.844   0.144
2     2    0.866   0.176
3     3    0.844   0.202
4     4    0.837   0.164
fold       2.000
roc_auc    0.845
pr_auc     0.170
dtype: float64


In [15]:
import torch
import lightning as L
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from lightning.pytorch.callbacks import EarlyStopping
from sklearn.model_selection import GroupShuffleSplit

L.seed_everything(42)   # same random start every run, so results repeat


Seed set to 42


42

In [16]:
class LitLogReg(L.LightningModule):
    def __init__(self, n_features, pos_weight, lr=1e-2, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.linear = nn.Linear(n_features, 1)          # one weight per column + a bias
        self.loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(float(pos_weight), dtype=torch.float32))

    def forward(self, x):
        return self.linear(x).squeeze(1)                # the risk score (before the S-curve)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)                 # how wrong were we?
        self.log("train_loss", loss)
        return loss                                     # Lightning nudges the weights using this

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.log("val_loss", self.loss_fn(self(x), y))  # used by early stopping

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)


In [17]:
def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=1024, shuffle=shuffle)


In [18]:
results_lit = []
for fold in range(5):
    tr = (df2.fold_group != fold).values
    va = (df2.fold_group == fold).values

    # Early stopping needs its own held-out loans. Take 10% of the TRAINING
    # customers, so the test fold stays completely unseen.
    fit_idx, stop_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
                             .split(X2[tr], y2[tr], df2.customer_id[tr]))
    X_fit, y_fit = X2[tr].iloc[fit_idx], y2[tr].iloc[fit_idx]
    X_stop, y_stop = X2[tr].iloc[stop_idx], y2[tr].iloc[stop_idx]

    # Same cleaning as before, learned from training rows only
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), num2),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat2),
    ]).fit(X_fit)
    A_fit, A_stop, A_va = pre.transform(X_fit), pre.transform(X_stop), pre.transform(X2[va])

    pos_weight = (y_fit == 0).sum() / (y_fit == 1).sum()      # about 54
    model = LitLogReg(A_fit.shape[1], pos_weight)

    trainer = L.Trainer(accelerator="cpu", max_epochs=50,
                        callbacks=[EarlyStopping(monitor="val_loss", patience=5)],
                        enable_progress_bar=False, logger=False,
                        enable_checkpointing=False, deterministic=True)
    trainer.fit(model, make_loader(A_fit, y_fit.values, True),
                       make_loader(A_stop, y_stop.values, False))

    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(torch.tensor(A_va, dtype=torch.float32))).numpy()

    results_lit.append({"fold": fold, "epochs": trainer.current_epoch,
                        "roc_auc": roc_auc_score(y2[va], p),
                        "pr_auc": average_precision_score(y2[va], p)})

results_lit = pd.DataFrame(results_lit)
print(results_lit.round(3))
print(results_lit.mean().round(3))


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/hiro/Desktop/Ashesi_School/Courses/Year1_First_Semester/Machine_Learning_Essentials/Prosits/mle-group-2-prosit-2/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name    | Type              | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | linear  | Linear            | 49     | train | 0    
1 | loss_fn | BCEWithLogitsLoss | 0      | train | 0    
--------------------------------------------------------------
49        Trainable params
0         Non-trainable params
49        Total params
0.000     Total estimated model pa

   fold  epochs  roc_auc  pr_auc
0     0      17    0.836   0.152
1     1       8    0.863   0.151
2     2       8    0.852   0.170
3     3      11    0.837   0.211
4     4      17    0.837   0.165
fold        2.000
epochs     12.200
roc_auc     0.845
pr_auc      0.170
dtype: float64


In [19]:
pd.DataFrame({
    "scikit-learn": results2[["roc_auc", "pr_auc"]].mean(),
    "Lightning": results_lit[["roc_auc", "pr_auc"]].mean(),
}).round(3)


,scikit-learn,Lightning
roc_auc,0.845,0.845
pr_auc,0.170,0.170
